In [1]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from netneurotools.networks import struct_consensus, binarize_network


In [2]:
# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
matlab_conns_path = "/Users/adrian/Documents/01_projects/14_4D_lab/connectome_distances/data/raw/lexis_data/connectomes.mat"
age_path          = "/Users/adrian/Documents/01_projects/14_4D_lab/connectome_distances/data/raw/lexis_data/age.mat"
ids_path          = "/Users/adrian/Documents/01_projects/14_4D_lab/connectome_distances/data/raw/lexis_data/ids.csv"
distance_path     = "/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/data/preprocessed/lexis_data/02_distance_matrices/distance_matrix_100.npy"
base_path         = Path("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/data/preprocessed")

N_NODES  = 100
DENSITY  = 10   # percent density for binarization

PER_AGE_BIN = 1  # 1 for one-year bins, 2 for two-year bins

In [3]:
# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
conns_mat = h5py.File(matlab_conns_path)
age_mat   = h5py.File(age_path)
ids_df    = pd.read_csv(ids_path)

conns_all = np.array(conns_mat['connectomes']).T          # (n_subjects, 100, 100)
ages_all  = np.array(age_mat["age"])[0]                   # (n_subjects,)

d      = np.load(distance_path)                           # (100, 100)
hemiid = np.array([0]*50 + [1]*50).reshape(-1, 1)


In [4]:
# ─────────────────────────────────────────────
# DEFINE PARTICIPANT GROUPS  (same as script 2)
# ─────────────────────────────────────────────
id_col = ids_df.columns[0]   # first column holds the IDs

participant_groups = {
    "young":      ids_df[~ids_df[id_col].str.startswith('HC')].index,
    "aging":      ids_df[ids_df[id_col].str.startswith('HCA')].index,
    "developing": ids_df[ids_df[id_col].str.startswith('HCD')].index,
}

for name, idx in participant_groups.items():
    print(f"{name.capitalize()}: {len(idx)} participants")


Young: 1065 participants
Aging: 718 participants
Developing: 635 participants


In [5]:
# ─────────────────────────────────────────────
# HELPER: consensus network for a set of connectomes
# Falls back to the single matrix when n==1,
# and to a mean when struct_consensus cannot run.
# ─────────────────────────────────────────────
def compute_consensus(conns_subset: np.ndarray) -> np.ndarray:
    """
    conns_subset : (n_subjects, N_NODES, N_NODES)
    returns       : binarized consensus (N_NODES, N_NODES)
    """
    n = conns_subset.shape[0]
    if n == 1 or n == 2:
        # raw = conns_subset[0]
        print("ERROR: struct_consensus needs ≥ 3 subjects; cannot compute consensus for n =", n)
        return None
    
    # elif n == 2:
    #     # struct_consensus needs ≥ 3 subjects; fall back to mean
    #     raw = conns_subset.mean(axis=0)
    else:
        raw = struct_consensus(conns_subset.T, d, hemiid=hemiid, weighted=True)
    
    return binarize_network(raw, retain=DENSITY)   # (N_NODES, N_NODES)

In [6]:
# ─────────────────────────────────────────────
# HELPER: build age-stratified consensus stack
# Returns:
#   consensus_stack  (N_NODES, N_NODES, n_ages)
#   age_labels       list of integer ages (length n_ages)
# ─────────────────────────────────────────────
def age_stratified_consensus(conns: np.ndarray, ages: np.ndarray):
    """
    conns : (n_subjects, N_NODES, N_NODES)
    ages  : (n_subjects,)  – continuous, will be rounded to nearest int
    """
    ages_rounded  = np.round(ages).astype(int)
    
    if PER_AGE_BIN == 1:
         unique_ages = np.sort(np.unique(ages_rounded))
    elif PER_AGE_BIN == 2:
        ages_binned = (np.floor(ages).astype(int) // 2) * 2  # e.g. 6,7 → 6; 8,9 → 8
        unique_ages = np.sort(np.unique(ages_binned))
    else: 
        raise ValueError("Invalid PER_AGE_BIN value. Use 1 for one-year bins or 2 for two-year bins.")
    
    n_ages        = len(unique_ages)

    consensus_stack = np.zeros((N_NODES, N_NODES, n_ages), dtype=np.float32)

    for i, age in enumerate(unique_ages):
        mask   = ages_rounded == age
        subset = conns[mask]          # (n_at_age, N_NODES, N_NODES)
        n_at   = subset.shape[0]
        print(f"  age {age:3d}: {n_at:3d} subject(s)")
        consensus_stack[:, :, i] = compute_consensus(subset)

    # -> (n_ages, N_NODES, N_NODES) for saving
    consensus_stack = consensus_stack.T # np.transpose(consensus_stack, (2, 0, 1))
    
    return consensus_stack, unique_ages.tolist()



In [7]:


# ─────────────────────────────────────────────
# PER-DATASET: consensus per age
# ─────────────────────────────────────────────
all_conns_list = []   # collect for the cross-dataset step
all_ages_list  = []

for group_name, idx in participant_groups.items():
    print(f"\n── {group_name.upper()} ──")
    conns_group = conns_all[idx]          # (n_group, 100, 100)
    ages_group  = ages_all[idx]

    all_conns_list.append(conns_group)
    all_ages_list.append(ages_group)

    consensus_stack, age_labels = age_stratified_consensus(conns_group, ages_group)
    # consensus_stack shape: (n_ages, N_NODES, N_NODES) 
    
    # save
    out_dir = base_path / f"lexis_data_{group_name}_consensus_per_age_{PER_AGE_BIN}_year" / "01_connectomes"
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "00_individual_connectomes_bin.npy", consensus_stack)
    
    out_dir = base_path / f"lexis_data_{group_name}_consensus_per_age_{PER_AGE_BIN}_year" / "04_further_info"
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "00_ages.npy", np.array(age_labels))
    
    out_dir = base_path / f"lexis_data_{group_name}_consensus_per_age_{PER_AGE_BIN}_year" / "02_distance_matrices"
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "distance_matrix_100.npy", d)

    print(f"  Saved consensus_per_age.npy  shape {consensus_stack.shape}")
    print(f"  Ages: {age_labels}")

    # quick sanity plot
    mid = consensus_stack.shape[0] // 2
    plt.figure(figsize=(4, 4))
    plt.imshow(consensus_stack[mid, :, :], cmap='hot', interpolation='nearest')
    plt.title(f"{group_name} - median age ({age_labels[mid]})")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(out_dir / f"consensus_per_age_midslice.png", dpi=100)
    plt.close()



── YOUNG ──
  age  20:   1 subject(s)
ERROR: struct_consensus needs ≥ 3 subjects; cannot compute consensus for n = 1
  age  22:  50 subject(s)
  age  23:  61 subject(s)
  age  24:  46 subject(s)
  age  25:  67 subject(s)
  age  26:  89 subject(s)
  age  27: 106 subject(s)
  age  28:  86 subject(s)
  age  29:  95 subject(s)
  age  30:  90 subject(s)
  age  31:  84 subject(s)
  age  32:  92 subject(s)
  age  33:  84 subject(s)
  age  34:  54 subject(s)
  age  35:  50 subject(s)
  age  36:   9 subject(s)
  age  37:   1 subject(s)
ERROR: struct_consensus needs ≥ 3 subjects; cannot compute consensus for n = 1
  Saved consensus_per_age.npy  shape (17, 100, 100)
  Ages: [20, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]

── AGING ──
  age  28:   1 subject(s)
ERROR: struct_consensus needs ≥ 3 subjects; cannot compute consensus for n = 1
  age  36:   9 subject(s)
  age  37:  14 subject(s)
  age  38:  20 subject(s)
  age  39:  17 subject(s)
  age  40:  10 subject(s)
  age  41:

In [8]:


# ─────────────────────────────────────────────
# CROSS-DATASET: one consensus network per age
# (pool all three groups together)
# ─────────────────────────────────────────────
print("\n── ALL DATASETS COMBINED ──")

conns_combined = np.concatenate(all_conns_list, axis=0)  # (total_n, 100, 100)
ages_combined  = np.concatenate(all_ages_list,  axis=0)  # (total_n,)

print(f"  Total subjects: {conns_combined.shape[0]}")

consensus_stack_all, age_labels_all = age_stratified_consensus(conns_combined, ages_combined)

# save
out_dir_all = base_path / f"lexis_data_all_consensus_per_age_{PER_AGE_BIN}_year" / "01_connectomes"
out_dir_all.mkdir(parents=True, exist_ok=True)
np.save(out_dir_all / "00_individual_connectomes_bin.npy",        consensus_stack_all)

out_dir = base_path / f"lexis_data_all_consensus_per_age_{PER_AGE_BIN}_year" / "04_further_info"
out_dir.mkdir(parents=True, exist_ok=True)
np.save(out_dir / "00_ages.npy", np.array(age_labels_all))

out_dir = base_path / f"lexis_data_all_consensus_per_age_{PER_AGE_BIN}_year" / "02_distance_matrices"
out_dir.mkdir(parents=True, exist_ok=True)
np.save(out_dir / "distance_matrix_100.npy", d)

print(f"  Saved consensus_per_age.npy  shape {consensus_stack_all.shape}")
print(f"  Ages spanned: {age_labels_all[0]} – {age_labels_all[-1]}")

# quick sanity plot
mid = consensus_stack_all.shape[0] // 2
plt.figure(figsize=(4, 4))
plt.imshow(consensus_stack_all[mid, :, :], cmap='hot', interpolation='nearest')
plt.title(f"All datasets - median age ({age_labels_all[mid]})")
plt.colorbar()
plt.tight_layout()
plt.savefig(out_dir_all / "consensus_per_age_midslice.png", dpi=100)
plt.close()

print("\nDone. Output layout:")
print("  <base_path>/lexis_data_<group>/01_connectomes/consensus_per_age.npy       → (100, 100, n_ages)")
print("  <base_path>/lexis_data_<group>/01_connectomes/consensus_per_age_labels.npy → (n_ages,)  integer years")
print("  <base_path>/lexis_data_all/01_connectomes/  same files, pooled across all groups")


── ALL DATASETS COMBINED ──
  Total subjects: 2418
  age   6:   5 subject(s)
  age   7:   5 subject(s)
  age   8:  20 subject(s)
  age   9:  51 subject(s)
  age  10:  50 subject(s)
  age  11:  42 subject(s)
  age  12:  45 subject(s)
  age  13:  45 subject(s)
  age  14:  66 subject(s)
  age  15:  62 subject(s)
  age  16:  43 subject(s)
  age  17:  38 subject(s)
  age  18:  30 subject(s)
  age  19:  40 subject(s)
  age  20:  26 subject(s)
  age  21:  47 subject(s)
  age  22:  71 subject(s)
  age  23:  61 subject(s)
  age  24:  46 subject(s)
  age  25:  67 subject(s)
  age  26:  89 subject(s)
  age  27: 106 subject(s)
  age  28:  87 subject(s)
  age  29:  95 subject(s)
  age  30:  90 subject(s)
  age  31:  84 subject(s)
  age  32:  92 subject(s)
  age  33:  84 subject(s)
  age  34:  54 subject(s)
  age  35:  50 subject(s)
  age  36:  18 subject(s)
  age  37:  15 subject(s)
  age  38:  20 subject(s)
  age  39:  17 subject(s)
  age  40:  10 subject(s)
  age  41:  26 subject(s)
  age  42:  